# Complete FastHTML Application Example

This notebook demonstrates a complete application with FastHTML's SSE support, HTMX integration, multiple endpoints, error handling, and job status tracking.

In [1]:
from fasthtml.common import *
import uuid, time, json

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream
from cjm_tqdm_capture.progress_info import serialize_all_jobs

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import gap
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from cjm_fasthtml_daisyui.core.testing import create_test_app

app, rt = create_test_app(theme=DaisyUITheme.DARK)

# Initialize with history for debugging
monitor = ProgressMonitor(keep_history=True, history_limit=100)
runner = JobRunner(monitor)

# Store job results - renamed to avoid conflict with endpoint function
job_results_store = {}

In [4]:
# Define a complex task with parameters and multiple stages
def complex_task(task_name, iterations, speed="normal"):
    from tqdm import tqdm
    import time
    import random
    
    speeds = {"fast": 0.001, "normal": 0.01, "slow": 0.05}
    delay = speeds.get(speed, 0.01)
    
    results = {"task": task_name, "stages": []}
    
    # Stage 1: Initialization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Initialize"):
        time.sleep(delay)
    results["stages"].append("Initialization complete")
    
    # Stage 2: Processing
    processed_data = []
    for i in tqdm(range(iterations), desc=f"{task_name}: Process"):
        time.sleep(delay)
        processed_data.append(i * random.random())
    results["stages"].append(f"Processed {len(processed_data)} items")
    
    # Stage 3: Validation
    for i in tqdm(range(iterations // 2), desc=f"{task_name}: Validate"):
        time.sleep(delay)
    results["stages"].append("Validation complete")
    
    # Stage 4: Finalization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Finalize"):
        time.sleep(delay)
    results["stages"].append("Finalization complete")
    
    results["summary"] = f"Task {task_name} completed successfully with {iterations} iterations"
    return results

In [5]:
# Enhanced UI with smart polling and precise button updates
from cjm_fasthtml_daisyui.core.testing import create_test_page

def render_start_button(disabled=False, oob_swap=False):
    """Render just the Start Task button with appropriate state
    
    Args:
        disabled: Whether the button should be disabled
        oob_swap: Whether to include OOB swap attribute for HTMX
    """
    btn_class = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'type': 'submit',
        'id': 'start-btn',
        'cls': btn_class,
        'disabled': disabled
    }
    
    if oob_swap:
        kwargs['hx_swap_oob'] = 'true'
    
    return Button("Start Task", **kwargs)

def render_clear_button(disabled=False, oob_swap=False):
    """Render just the Clear Completed button with appropriate state
    
    Args:
        disabled: Whether the button should be disabled  
        oob_swap: Whether to include OOB swap attribute for HTMX
    """
    btn_class = combine_classes(
        btn,
        btn_colors.secondary,
        m.l(2),
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'type': 'button',
        'id': 'clear-btn',
        'hx_post': '/clear_jobs',
        'hx_target': '#jobs-container',
        'hx_swap': 'innerHTML',
        'cls': btn_class,
        'disabled': disabled
    }
    
    if oob_swap:
        kwargs['hx_swap_oob'] = 'true'
    
    return Button("Clear Completed", **kwargs)

def render_button_group(start_disabled=False, clear_disabled=False):
    """Render the button group without the form wrapper
    
    Args:
        start_disabled: Whether the Start Task button should be disabled
        clear_disabled: Whether the Clear Completed button should be disabled
    """
    return Div(
        render_start_button(disabled=start_disabled),
        render_clear_button(disabled=clear_disabled),
        id="button-group",
        cls=combine_classes("flex", gap(2))
    )

def render_task_form(submit_disabled=False):
    """Render the complete task configuration form
    
    Args:
        submit_disabled: Whether the submit button should be disabled
    """
    return Form(
        H2("Configure Task", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
        Div(
            Label("Task Name:", cls=str(label), fr="task-name"),
            Input(id="task-name", name="task_name", type="text", value="MyTask", 
                  cls=combine_classes(text_input, w.full, max_w.xs)),
            cls=str(m.b(4))
        ),
        Div(
            Label("Iterations:", cls=str(label), fr="iterations"),
            Input(id="iterations", name="iterations", type="number", value="100", 
                  min="10", max="500", cls=combine_classes(text_input, w.full, max_w.xs)),
            cls=str(m.b(4))
        ),
        Div(
            Label("Speed:", cls=str(label), fr="speed"),
            Select(
                Option("Fast", value="fast"),
                Option("Normal", value="normal", selected=True),
                Option("Slow", value="slow"),
                id="speed",
                name="speed",
                cls=combine_classes(select, w.full, max_w.xs)
            ),
            cls=str(m.b(4))
        ),
        render_button_group(start_disabled=submit_disabled, clear_disabled=False),
        hx_post="/create_job",
        hx_target="#progress-container",
        hx_swap="innerHTML",
        cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6)),
        id="task-form"
    )

@rt("/")
def index():
    return create_test_page(
        "Complete Progress Application",
        Div(
            H1("Task Manager with Progress Tracking", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Task configuration form
            render_task_form(submit_disabled=False),
            
            # Progress display container
            Div(
                H2("Current Progress", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    P("No active job", cls=str(font_size.sm)),
                    id="progress-container"
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Active jobs list - with smart polling
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Div("Loading jobs...", id="jobs-list"),
                    hx_get="/jobs_table",
                    hx_trigger="load",  # Only load initially, smart polling handled by jobs_table
                    hx_swap="innerHTML",
                    id="jobs-container",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [6]:
# Helper functions for rendering UI elements with polling control
def render_hidden_trigger(endpoint, target_id, swap="outerHTML", trigger="load"):
    """Render a hidden div that triggers an HTMX update
    
    Args:
        endpoint: The endpoint to call (e.g., "/job_progress_cell")
        target_id: The target element ID to update (e.g., "#progress-123")
        swap: The HTMX swap strategy (default: "outerHTML")
        trigger: The HTMX trigger (default: "load")
    """
    from cjm_fasthtml_tailwind.utilities.layout import display_tw
    return Div(
        hx_get=endpoint,
        hx_trigger=trigger,
        hx_target=target_id,
        hx_swap=swap,
        style="display: none"
    )

def render_job_progress_cell(job_id, progress_value, enable_polling=False):
    """Render a job progress cell with optional polling
    
    Args:
        job_id: The job ID
        progress_value: The current progress value (0-100)
        enable_polling: Whether to enable polling for updates
    """
    kwargs = {
        'id': f'progress-cell-{job_id}'
    }
    
    if enable_polling:
        kwargs['hx_get'] = f"/job_progress_cell?job_id={job_id}"
        kwargs['hx_trigger'] = "load delay:500ms"
        kwargs['hx_swap'] = "innerHTML"
    
    return Td(f"{progress_value:.1f}%", **kwargs)

def render_job_status_cell(job_id, status_text, enable_polling=False):
    """Render a job status cell with optional polling
    
    Args:
        job_id: The job ID
        status_text: The status text to display
        enable_polling: Whether to enable polling for updates
    """
    kwargs = {
        'id': f'status-cell-{job_id}'
    }
    
    if enable_polling:
        kwargs['hx_get'] = f"/job_status_cell?job_id={job_id}"
        kwargs['hx_trigger'] = "load delay:500ms"
        kwargs['hx_swap'] = "innerHTML"
    
    return Td(status_text, **kwargs)

def render_job_action_cell(job_id, job_completed=False):
    """Render a job action cell
    
    Args:
        job_id: The job ID
        job_completed: Whether the job is completed
    """
    view_button = Button(
        "View",
        hx_get=f"/view_job?job_id={job_id}",
        hx_target="#progress-container",
        hx_swap="innerHTML transition:true",
        cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary if not job_completed else "")
    )
    
    return Td(view_button, id=f'action-cell-{job_id}')

def render_job_row(job_id, job_data, serialized_data):
    """Render a complete job row with cell-level polling
    
    Args:
        job_id: The job ID
        job_data: The job data from monitor
        serialized_data: The serialized job data
    """
    is_running = not serialized_data['completed']
    
    return Tr(
        Td(job_id[:8] + "...", id=f"id-cell-{job_id}"),
        render_job_progress_cell(job_id, serialized_data['overall_progress'], enable_polling=is_running),
        render_job_status_cell(job_id, "Running" if is_running else "Complete", enable_polling=is_running),
        render_job_action_cell(job_id, job_completed=serialized_data['completed']),
        id=f"job-row-{job_id}"
    )

In [7]:
# API endpoints with smart polling and precise button updates
@rt("/create_job")
def create_job(task_name: str, iterations: int, speed: str):
    job_id = str(uuid.uuid4())
    
    # Wrapper to store results
    def task_wrapper():
        result = complex_task(task_name, iterations, speed)
        job_results_store[job_id] = result
        return result
    
    # Adjust throttling based on speed
    patch_config = {
        'fast': {"min_delta_pct": 10, "min_update_interval": 0.01, "emit_initial": True},
        'normal': {"min_delta_pct": 5, "min_update_interval": 0.05, "emit_initial": True},
        'slow': {"min_delta_pct": 2, "min_update_interval": 0.2, "emit_initial": True}
    }
    
    runner.start(
        job_id,
        task_wrapper,
        patch_kwargs=patch_config.get(speed, patch_config['normal'])
    )
    
    # Small delay to allow monitor to register the job
    import time
    time.sleep(0.1)
    
    # Get updated jobs table for OOB swap
    jobs_table_update = jobs_table()
    
    # Return response with OOB swaps
    return Div(
        # Progress display for #progress-container
        Div(
            H3(f"Task: {task_name}", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(2))),
            P(f"Job ID: {job_id[:8]}...", cls=combine_classes(font_size.xs, m.b(4))),
            
            # Progress section with smart polling
            Div(
                # Initial state
                P("Overall: 0%", cls=str(font_weight.bold)),
                Progress(value="0", max="100", 
                        cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))),
                P("Starting...", cls=str(font_size.sm)),
                
                # HTMX polling for progress
                hx_get=f"/job_progress?job_id={job_id}",
                hx_trigger="load delay:100ms",
                hx_swap="innerHTML",
                id=f"progress-{job_id}"
            ),
            
            # Results section
            Div(
                hx_get=f"/job_results_section?job_id={job_id}",
                hx_trigger="load",
                hx_swap="outerHTML",
                id=f"results-{job_id}"
            )
        ),
        
        # OOB swaps - only update the start button, not the entire form
        render_start_button(disabled=True, oob_swap=True),  # Disable only the start button
        Div(
            jobs_table_update,
            id="jobs-container",
            hx_swap_oob="true",
            cls=str(overflow.x.auto)
        )
    )

@rt("/job_progress")
def job_progress(job_id: str):
    """Returns current progress for a job with conditional polling"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        # Job not found or not started yet - continue polling
        return Div(
            P("Waiting for job to start...", cls=str(font_weight.bold)),
            Progress(value="0", max="100", 
                    cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))),
            # Keep polling until job starts
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load delay:100ms",
            hx_swap="outerHTML"
        )
    
    # Build progress bars dynamically based on actual tqdm bars
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                  cls=combine_classes(font_size.sm, font_weight.semibold)),
                Progress(
                    value=str(bar.progress), 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(2))
            )
        )
    
    # Check if completed
    if snapshot['completed']:
        # Check if any other jobs are running
        all_jobs = monitor.all()
        has_running = any(not job['completed'] for job in all_jobs.values())
        
        # Re-enable start button if no other active jobs via OOB swap
        button_update = render_start_button(disabled=has_running, oob_swap=True) if not has_running else ""
        
        # Return final progress state without polling, with OOB button update
        return Div(
            # Final progress display
            P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
            Progress(
                value=str(snapshot['overall_progress']), 
                max="100", 
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
            *bars_html,
            P("Task completed!", cls=combine_classes(font_size.sm, font_weight.semibold, m.t(2))),
            # Include OOB button update if needed
            button_update
        )
    
    # Task is in progress - continue polling
    return Div(
        P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
        Progress(
            value=str(snapshot['overall_progress']), 
            max="100", 
            cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
        ),
        *bars_html,
        P(f"Running... ({len(snapshot['bars'])} active bars)", cls=str(font_size.sm)),
        # Continue polling while in progress
        hx_get=f"/job_progress?job_id={job_id}",
        hx_trigger="load delay:100ms",
        hx_swap="outerHTML"
    )

@rt("/job_results_section")
def job_results_section(job_id: str):
    """Returns the results section with conditional polling"""
    snapshot = monitor.snapshot(job_id)
    
    # If job not complete, keep polling
    if not snapshot or not snapshot['completed']:
        return Div(
            # Empty while running, check again later
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load delay:500ms",  # Poll less frequently for results
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    
    # Job is complete - check for results
    if job_id not in job_results_store:
        # Still waiting for results to be stored
        return Div(
            P("Processing results...", cls=str(font_size.sm)),
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    
    # Results are available - return static div (no more polling)
    return Div(
        Details(
            Summary("View Results", cls=combine_classes(font_weight.semibold, m.t(4))),
            Pre(
                json.dumps(job_results_store[job_id], indent=2),
                cls=combine_classes(bg_dui.base_300, p(4), rounded(), font_size.xs, m.t(2))
            ),
            open=False
        ),
        id=f"results-{job_id}"  # Same ID, but no HTMX attributes = no more polling
    )

In [8]:
# Jobs table endpoint with smart polling and button state management
import json  # Add import for JSON serialization

@rt("/jobs_table")
def jobs_table():
    jobs = monitor.all()
    serialized = serialize_all_jobs(jobs)
    
    if not serialized:
        return Div("No active jobs", id="jobs-list")
    
    # Check if any jobs are running
    has_running = any(not job_data['completed'] for job_data in serialized.values())
    
    # Create table with jobs using cell-level updates
    rows = []
    for job_id, job_data in serialized.items():
        job = jobs[job_id]
        rows.append(render_job_row(job_id, job, job_data))
    
    table_elem = Table(
        Thead(
            Tr(
                Th("Job ID"),
                Th("Progress"),
                Th("Status"),
                Th("Action")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_modifiers.zebra, w.full),
        id="jobs-list"
    )
    
    # Only add polling to the table container if jobs are running
    if has_running:
        # Add a hidden trigger to refresh the table structure periodically
        # This will pick up new jobs but won't update existing job cells (they poll themselves)
        return Div(
            table_elem,
            render_hidden_trigger("/check_new_jobs", "#jobs-container", swap="none", trigger="load delay:2s")
        )
    
    return table_elem

@rt("/check_new_jobs")
def check_new_jobs():
    """Check if new jobs have been added and trigger table refresh if needed"""
    # This endpoint doesn't return content, just triggers a refresh if needed
    # You could add logic here to only refresh if the job count changed
    return Div(
        render_hidden_trigger("/jobs_table", "#jobs-list", trigger="load"),
        style="display: none"
    )

@rt("/job_progress_cell")
def job_progress_cell(job_id: str):
    """Update only the progress cell for a specific job"""
    jobs = monitor.all()
    serialized = serialize_all_jobs(jobs)
    
    if job_id not in serialized:
        return ""
    
    job_data = serialized[job_id]
    
    # If still running, return cell with continued polling
    if not job_data['completed']:
        return Div(
            f"{job_data['overall_progress']:.1f}%",
            hx_get=f"/job_progress_cell?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML"
        )
    else:
        # Job completed, return static text
        return f"{job_data['overall_progress']:.1f}%"

@rt("/job_status_cell")
def job_status_cell(job_id: str):
    """Update only the status cell for a specific job"""
    jobs = monitor.all()
    serialized = serialize_all_jobs(jobs)
    
    if job_id not in serialized:
        return ""
    
    job_data = serialized[job_id]
    status_text = "Complete" if job_data['completed'] else "Running"
    
    # If still running, return cell with continued polling
    if not job_data['completed']:
        return Div(
            status_text,
            hx_get=f"/job_status_cell?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML"
        )
    else:
        # Job completed, return static text and trigger button state check
        return Div(
            status_text,
            # Trigger button state check when job completes
            render_hidden_trigger("/check_button_state", "#start-btn", swap="outerHTML"),
            # Trigger an update to refresh the table polling state
            render_hidden_trigger("/check_table_polling", "#jobs-container", swap="none")
        )

@rt("/check_table_polling")
def check_table_polling():
    """Check if table polling should stop"""
    jobs = monitor.all()
    serialized = serialize_all_jobs(jobs)
    has_running = any(not job_data['completed'] for job_data in serialized.values())
    
    if not has_running:
        # All jobs completed, stop table polling by returning empty div
        return Div(style="display: none")
    
    # Jobs still running, continue checking
    return render_hidden_trigger("/check_table_polling", "#jobs-container", swap="none", trigger="load delay:2s")

@rt("/view_job")
def view_job(job_id: str):
    """View a specific job's progress with conditional polling"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        return P("Job not found", cls=combine_classes(font_size.sm))
    
    # Pre-populate with actual current progress to avoid empty state
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                  cls=combine_classes(font_size.sm, font_weight.semibold)),
                Progress(
                    value=str(bar.progress), 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(2))
            )
        )
    
    # Pre-render results section based on current state
    if snapshot['completed'] and job_id in job_results_store:
        # Job is complete and results are available
        results_section = Div(
            Details(
                Summary("View Results", cls=combine_classes(font_weight.semibold, m.t(4))),
                Pre(
                    json.dumps(job_results_store[job_id], indent=2),
                    cls=combine_classes(bg_dui.base_300, p(4), rounded(), font_size.xs, m.t(2))
                ),
                open=False
            ),
            id=f"results-{job_id}"  # Static div, no polling
        )
    elif snapshot['completed']:
        # Job is complete but results not yet available
        results_section = Div(
            P("Processing results...", cls=str(font_size.sm)),
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    else:
        # Job is still running - start polling for results
        results_section = Div(
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load",  # Single initial load
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    
    # Return everything pre-rendered
    return Div(
        H3(f"Viewing Job", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(2))),
        P(f"Job ID: {job_id[:8]}...", cls=combine_classes(font_size.xs, m.b(4))),
        
        # Progress section - conditional polling based on completion status
        Div(
            # Show actual current progress immediately
            P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
            Progress(
                value=str(snapshot['overall_progress']), 
                max="100", 
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
            *bars_html,
            P("Task completed!" if snapshot['completed'] else f"Running... ({len(snapshot['bars'])} active bars)", 
              cls=str(font_size.sm)),
            
            # HTMX polling only if not completed
            hx_get=f"/job_progress?job_id={job_id}" if not snapshot['completed'] else None,
            hx_trigger="load delay:100ms" if not snapshot['completed'] else None,
            hx_swap="innerHTML" if not snapshot['completed'] else None,
            id=f"progress-{job_id}"
        ),
        
        # Results section - pre-rendered based on state
        results_section
    )

In [9]:
# Get specific job status (for API usage)
@rt("/job_status")
def job_status(job_id: str):
    snapshot = monitor.snapshot(job_id)
    if not snapshot:
        return JSONResponse({"error": "Job not found"}, status_code=404)
    
    return JSONResponse({
        "job_id": job_id,
        "progress": snapshot["overall_progress"],
        "completed": snapshot["completed"],
        "bars": {
            k: {
                "description": v.description,
                "progress": v.progress,
                "current": v.current,
                "total": v.total
            }
            for k, v in snapshot["bars"].items()
        }
    })

In [10]:
# Get job result endpoint (for direct API usage)
@rt("/job_result")  
def job_result(job_id: str):
    """API endpoint to get job results as JSON"""
    if job_id in job_results_store:
        return JSONResponse(job_results_store[job_id])
    return JSONResponse({"error": "No results yet"}, status_code=404)

In [11]:
# SSE stream endpoint using FastHTML's EventStream
@rt("/stream")
def stream(job_id: str):
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.1,
            heartbeat=10.0,
            wait_for_start=True,
            start_timeout=30.0
        )
    )

In [12]:
# Clear completed jobs endpoint with precise button updates
@rt("/clear_jobs")
def clear_jobs():
    # Clear completed jobs from monitor
    monitor.clear_completed(older_than_seconds=0)
    
    # Also clear results for completed jobs
    for job_id in list(job_results_store.keys()):
        snapshot = monitor.snapshot(job_id)
        if not snapshot:
            del job_results_store[job_id]
    
    # Check if any jobs are still running
    all_jobs = monitor.all()
    has_running = any(not job['completed'] for job in all_jobs.values())
    
    # Get updated jobs table
    updated_table = jobs_table()
    
    # Return updated table with OOB button update if needed
    if not has_running:
        # No running jobs, enable the start button only via OOB swap
        return Div(
            updated_table,
            render_start_button(disabled=False, oob_swap=True)  # Enable only the start button
        )
    else:
        # Still have running jobs, keep start button disabled
        return updated_table

In [13]:
# Button state check endpoint
@rt("/check_button_state")
def check_button_state():
    """Check and update button states based on job status"""
    all_jobs = monitor.all()
    has_running = any(not job['completed'] for job in all_jobs.values())
    
    # Return start button with appropriate state
    return render_start_button(disabled=has_running, oob_swap=True)

In [14]:
# Start server for Jupyter
from cjm_fasthtml_daisyui.core.testing import start_test_server
from fasthtml.jupyter import HTMX
from IPython.display import display

server = start_test_server(app)
display(HTMX())

In [15]:
# Stop server when done
server.stop()